# PARC2026 — M2/M3 Common Model Benchmark Controller

D10で決まった **同じprovisional best dataset** を使い、
`π0.5 / SmolVLA / OpenVLA-OFT` を同じ評価契約で比較するためのcontrollerです。

正式比較は2系統です。

1. **Equal Data Exposure** — 同じepisode pool / sampling policy / comparable samples seen
2. **Equal Wall Time** — 同じA100 wall-time budget

このNotebookは **勝手に比較budgetを発明しません**。
`EQUAL_DATA_SAMPLE_BUDGET` と `EQUAL_WALL_TIME_SEC` を明示設定するまでM3を開始しません。

OpenVLA-OFTだけはRLDS入力が必要です。選択済みLeRobot subsetと同じepisode poolを
RLDSへ変換したartifactが無い場合は、M2で明示的にBLOCKします。
別datasetを代用して「公平な比較」と見なすことはしません。


In [ ]:
# 0/5 Preflight + D10 decision gate
import os, json, shutil, subprocess
from pathlib import Path
from google.colab import drive, userdata
drive.mount("/content/drive")
try:
    tok = os.environ.get("HF_TOKEN") or userdata.get("HF_TOKEN")
except Exception:
    tok = None
if not tok: raise RuntimeError("Colab Secretsに HF_TOKEN を登録してください。")
os.environ["HF_TOKEN"] = tok
gpu_name = subprocess.check_output(["nvidia-smi","--query-gpu=name","--format=csv,noheader"], text=True).strip()
vram = int(subprocess.check_output(["nvidia-smi","--query-gpu=memory.total","--format=csv,noheader,nounits"], text=True).strip())
print("GPU:", gpu_name, vram, "MiB")
if vram < 38000: raise RuntimeError("M2/M3 training bring-upはA100 40GB以上を使用してください。L4は推論feasibility評価用です。")
DRIVE = Path("/content/drive/MyDrive/parc2026-cache")
decision_path = DRIVE/"pi05-top2-tiebreak-v1/provisional_best_dataset_recipe.json"
assert decision_path.exists(), decision_path
decision = json.loads(decision_path.read_text())
if decision.get("status") != "DECIDED": raise RuntimeError(f"D10未決着: {decision}")
SELECTED_VARIANT = decision["selected_variant"]
print("SELECTED DATASET:", SELECTED_VARIANT)
print("=== D10 GATE: PASS ===")


In [ ]:
# 1/5 Clone repo + write pinned model bring-up manifest
import json, subprocess
from pathlib import Path
ROOT = Path("/content/parc2026"); REPO = ROOT/"py_AI"; ROOT.mkdir(parents=True, exist_ok=True)
if not (REPO/".git").exists(): subprocess.run(["git","clone","https://github.com/yu37330/py_AI.git",str(REPO)], check=True)
subprocess.run(["git","-C",str(REPO),"fetch","origin","main"], check=True)
subprocess.run(["git","-C",str(REPO),"checkout","--force","origin/main"], check=True)
DRIVE_OUT = Path("/content/drive/MyDrive/parc2026-cache/model-benchmark-v1"); DRIVE_OUT.mkdir(parents=True, exist_ok=True)
registry = json.loads((REPO/"experiments/model_registry_v1.json").read_text())
manifest = {"schema_version":1,"stage":"M2_candidate_bringup","selected_dataset_variant":SELECTED_VARIANT,"dataset_id":"lerobot/libero_plus","dataset_revision":"f3f49f426d75030177b18778374005bc12ccd588","models":registry["models"],"eval_contract":{"tracks_for_screening":["track1","track2"],"max_steps":300,"seed_set":[20260906,20260907],"same_task_selection":True,"same_n_episodes_per_task":True,"final_l4_24gb_inference_gate":True}}
(DRIVE_OUT/"model_bringup_manifest.json").write_text(json.dumps(manifest, indent=2)+"\n")
print(json.dumps(manifest, indent=2)); print("=== M2 MANIFEST: WRITTEN ===")


In [ ]:
# 2/5 Stage the same LeRobot source and verify selected group-aware manifest
import os, json, subprocess, sys
from pathlib import Path
DRIVE = Path("/content/drive/MyDrive/parc2026-cache"); src = DRIVE/"datasets/lerobot_libero_plus_v3_train"; dst = Path("/content/parc2026/datasets/public_libero_plus_v3_train")
assert (src/"meta/info.json").exists(), src
if not (dst/"meta/info.json").exists():
    dst.parent.mkdir(parents=True, exist_ok=True); print("=== Drive -> local dataset stage ===", flush=True)
    subprocess.run(["rsync","-a","--info=progress2",str(src)+"/",str(dst)+"/"], check=True)
else: print("dataset already staged:", dst)
stats = json.loads((dst/"meta/stats.json").read_text())
for f in ["observation.state","action"]: assert "q01" in stats[f] and "q99" in stats[f]
print("q01/q99: PASS")
env = os.environ.copy(); env.update({"PYTHONUNBUFFERED":"1","PARC_ROOT":"/content/parc2026","PY_AI_REPO":str(REPO),"PI05_DATASET_ROOT":str(dst),"PI05_DATASET_REPO_ID":"lerobot/libero_plus","PI05_DATASET_REVISION":"f3f49f426d75030177b18778374005bc12ccd588","RUN_ABLATIONS":"false"})
subprocess.run([sys.executable,"-u",str(REPO/"tools/colab/run_pi05_group_aware_ablation.py")], cwd=str(REPO), env=env, check=True)
mroot = Path("/content/parc2026/outputs/dataset_ablation_manifests_v2_group_aware")
name_map = {"V1_MULTI_FLAG_PRUNED_EXPERIMENTAL":"V1_MULTI_FLAG_PRUNED_EXPERIMENTAL.json","V2_SQRT_BALANCED_RAW":"V2_SQRT_BALANCED_RAW.json"}
selected_manifest = mroot/name_map[SELECTED_VARIANT]; assert selected_manifest.exists(), selected_manifest
selected = json.loads(selected_manifest.read_text()); assert selected["schema_version"] == 2 and selected["group_aware"] is True
SELECTED_MANIFEST=selected_manifest; EPISODE_COUNT=selected["summary"]["episode_count"]; FRAME_COUNT=selected["summary"]["frame_count"]
print("manifest:", SELECTED_MANIFEST); print("episodes:", EPISODE_COUNT, "frames:", FRAME_COUNT); print("=== COMMON DATASET GATE: PASS ===")


In [ ]:
# 3/5 Model bring-up + format gate
import json, os, subprocess
from pathlib import Path
ROOT=Path("/content/parc2026"); VENDOR=ROOT/"vendor"; VENDOR.mkdir(parents=True, exist_ok=True); DRIVE_OUT=Path("/content/drive/MyDrive/parc2026-cache/model-benchmark-v1")
pins={"smolvla":{"url":"https://github.com/huggingface/lerobot.git","sha":"3f2c29ef7e44b1ddccbcda3b6a63939e53639e9e","path":VENDOR/"lerobot-smolvla"},"openvla_oft":{"url":"https://github.com/small-zeng/openvla-oft.git","sha":"e4287e94541f459edc4feabc4e181f537cd569a8","path":VENDOR/"openvla-oft"}}
for name,cfg in pins.items():
    path=cfg["path"]
    if not (path/".git").exists():
        subprocess.run(["git","init","-q",str(path)],check=True); subprocess.run(["git","-C",str(path),"remote","add","origin",cfg["url"]],check=True)
    subprocess.run(["git","-C",str(path),"fetch","-q","--depth","1","origin",cfg["sha"]],check=True); subprocess.run(["git","-C",str(path),"checkout","-q","--force","FETCH_HEAD"],check=True)
    got=subprocess.check_output(["git","-C",str(path),"rev-parse","HEAD"],text=True).strip(); assert got==cfg["sha"],(name,got); print(name,"@",got)
assert (pins["smolvla"]["path"]/"src/lerobot").exists(); assert (pins["openvla_oft"]["path"]/"vla-scripts/finetune.py").exists()
rlds=Path(os.environ.get("OPENVLA_SELECTED_RLDS_ROOT","/content/drive/MyDrive/parc2026-cache/openvla-rlds-selected-v1")); rlds_contract=rlds/"conversion_contract.json"; openvla_ready=rlds_contract.exists()
status={"pi05":{"code":"PASS","dataset_format":"lerobot"},"smolvla":{"code":"PASS","dataset_format":"lerobot"},"openvla_oft":{"code":"PASS","dataset_format":"rlds","selected_subset_rlds_contract":"PASS" if openvla_ready else "MISSING","rlds_root":str(rlds)}}
(DRIVE_OUT/"bringup_status.json").write_text(json.dumps(status,indent=2)+"\n"); print(json.dumps(status,indent=2))
if not openvla_ready:
    print("\nM2 BLOCKED: OpenVLA-OFT requires RLDS for the SAME selected episode pool."); print("Run the LeRobot -> RLDS bridge before declaring M2 PASS.")
else:
    contract=json.loads(rlds_contract.read_text()); assert contract["selected_dataset_variant"]==SELECTED_VARIANT; assert contract["source_episode_ids_sha256"]==selected.get("episode_ids_sha256"); print("OpenVLA selected-subset RLDS contract: PASS")


In [ ]:
# 4/5 Freeze comparison budgets and job specs
import json
from pathlib import Path
EQUAL_DATA_SAMPLE_BUDGET = None
EQUAL_WALL_TIME_SEC = None
DRIVE_OUT=Path("/content/drive/MyDrive/parc2026-cache/model-benchmark-v1"); status=json.loads((DRIVE_OUT/"bringup_status.json").read_text())
protocol={"schema_version":1,"selected_dataset_variant":SELECTED_VARIANT,"selected_episode_count":EPISODE_COUNT,"selected_frame_count":FRAME_COUNT,"selected_episode_ids_sha256":selected.get("episode_ids_sha256"),"equal_data_exposure":{"sample_budget":EQUAL_DATA_SAMPLE_BUDGET,"same_episode_pool":True,"same_sampling_policy":True},"equal_wall_time":{"a100_wall_time_sec":EQUAL_WALL_TIME_SEC},"required_metrics":["simulator_success_rate","steps_to_success","episode_duration","inference_latency","peak_inference_vram","train_wall_time","peak_train_vram"],"promotion":{"max_models":2,"training_loss_alone_is_not_sufficient":True},"status":"BLOCKED"}
reasons=[]
if status["openvla_oft"]["selected_subset_rlds_contract"] != "PASS": reasons.append("openvla_selected_subset_rlds_missing")
if EQUAL_DATA_SAMPLE_BUDGET is None: reasons.append("equal_data_sample_budget_not_frozen")
if EQUAL_WALL_TIME_SEC is None: reasons.append("equal_wall_time_budget_not_frozen")
if not reasons: protocol["status"]="READY"
protocol["blocked_reasons"]=reasons; (DRIVE_OUT/"comparison_protocol.json").write_text(json.dumps(protocol,indent=2)+"\n"); print(json.dumps(protocol,indent=2))
if reasons: print("\nM3はまだ開始しません。上記Gateを解消後、このcellを再実行してください。")
else: print("=== M3 COMPARISON CONTRACT: READY ===")
